In [1]:
import getpass
import os

def _set_env(var: str):
    if not os.environ.get(var):
        os.environ[var] = getpass.getpass(f'{var}: ')
    
_set_env('OPENAI_API_KEY')
_set_env('ANTHROPIC_API_KEY')

In [8]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from langchain_core.tools import tool
import pandas as pd

@tool
def describe_data(csv: str) -> str:
    """Describe the data column in the dataframe.
    
    Args:
        csv: csv data path string
    """
    df = pd.read_csv(csv)
    describe_str = f'''Data: {csv}''' + df.describe(include='all').to_string()
    return describe_str

In [5]:
tools = [describe_data]

In [6]:
from langchain_openai import ChatOpenAI
from langchain_anthropic import ChatAnthropic

llm_gpt = ChatOpenAI(model='gpt-4o-mini')
llm_with_tools = llm_gpt.bind_tools(tools, tool_choice='any')

In [13]:
response = llm_with_tools.invoke(
    'https://raw.githubusercontent.com/pycaret/pycaret/master/datasets/diabetes.csv 이 데이터의 전처리를 해주세요.'
)

In [14]:
response.tool_calls[0]['args']

{'csv': 'https://raw.githubusercontent.com/pycaret/pycaret/master/datasets/diabetes.csv'}

In [15]:
response

AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_bLXNa7gXWT64ES8N2FTmWOWg', 'function': {'arguments': '{"csv":"https://raw.githubusercontent.com/pycaret/pycaret/master/datasets/diabetes.csv"}', 'name': 'describe_data'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 31, 'prompt_tokens': 81, 'total_tokens': 112, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_644f11dd4d', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019b3e9e-c23f-7b20-9e6e-4379aeb5cf34-0', tool_calls=[{'name': 'describe_data', 'args': {'csv': 'https://raw.githubusercontent.com/pycaret/pycaret/master/datasets/diabetes.csv'}, 'id': 'call_bLXNa7gXWT64ES8N2FTmWOWg', 'type': 'tool_call'}], usage_metadata={'input_tokens':

In [7]:
from pydantic import BaseModel, Field

class code(BaseModel):
    '''Schema for code solutions.'''

    prefix: str = Field(description='Description of the problem and approach')
    imports: str = Field(description='Code block import statements')
    code: str = Field(description='Code block not including import statements')

In [8]:
from langchain_core.prompts import ChatPromptTemplate

GENERATE_CODE_TEMPLATE = '''
Given the following pandas `describe()` output of a dataset,

write a **directly executable Python code** to:
1. handle missing values,
2. convert categorical columns,
3. ...any additional preprocessing needed,
4. prepare the dataset for machine learning.

Here is the describe result of the dataset:
\n ------- \n {context} \n ------- \n

Do not wrap the code in a function and the response in any backticks or anything else. The code should be written as a flat script, so that it can be run immediately and any errors will be visible during execution.
Ensure any code you provide can be executed \n
with all requried imports and variables defined. Structure your answer with a description of the code solution. \n
Then list the imports. And finally list the functioning code block.
'''

code_gen_prompt = ChatPromptTemplate.from_messages(
    [
        ('user', GENERATE_CODE_TEMPLATE),
    ]
)

In [9]:
from langchain_anthropic import ChatAnthropic

llm_claude = ChatAnthropic(model='claude-3-7-sonnet-20250219')

In [19]:
tool_result = describe_data.invoke(response.tool_calls[0]['args'])

In [20]:
print(tool_result)

Data: https://raw.githubusercontent.com/pycaret/pycaret/master/datasets/diabetes.csv       Number of times pregnant  Plasma glucose concentration a 2 hours in an oral glucose tolerance test  Diastolic blood pressure (mm Hg)  Triceps skin fold thickness (mm)  2-Hour serum insulin (mu U/ml)  Body mass index (weight in kg/(height in m)^2)  Diabetes pedigree function  Age (years)  Class variable
count                768.000000                                                                768.000000                        768.000000                        768.000000                      768.000000                                      768.000000                  768.000000   768.000000      768.000000
mean                   3.845052                                                                120.894531                         69.105469                         20.536458                       79.799479                                       31.992578                    0.471876    33.240885

In [21]:
generated_code = llm_claude.invoke(
    code_gen_prompt.format_messages(context=tool_result)
)
print('generated_code', generated_code)
code_structurer = llm_gpt.with_structured_output(code)
code_solution = code_structurer.invoke(generated_code.content)
print('code_solution', code_solution)

generated_code content='# Data Preprocessing for Diabetes Dataset\n\nThis code will preprocess the diabetes dataset for machine learning by:\n1. Handling missing values (which appear as 0s in certain medical measurements)\n2. Normalizing numerical features\n3. Performing final dataset preparation for machine learning\n\n## Imports\n\n```python\nimport pandas as pd\nimport numpy as np\nfrom sklearn.preprocessing import StandardScaler\nfrom sklearn.model_selection import train_test_split\n```\n\n## Preprocessing Code\n\n```python\n# Load the dataset\nurl = "https://raw.githubusercontent.com/pycaret/pycaret/master/datasets/diabetes.csv"\ndf = pd.read_csv(url)\n\n# Display basic information\nprint("Dataset shape:", df.shape)\nprint("Column names:", df.columns.tolist())\n\n# 1. Handle missing values (zeros in medical measurements)\n# Medical measurements that cannot be zero (except for \'Number of times pregnant\')\nzero_not_allowed_columns = [\n    \'Plasma glucose concentration a 2 hours 

In [22]:
code_solution

code(prefix='This code preprocesses the diabetes dataset for machine learning by handling missing values, normalizing numerical features, and preparing the final dataset for model training.', imports='import pandas as pd\nimport numpy as np\nfrom sklearn.preprocessing import StandardScaler\nfrom sklearn.model_selection import train_test_split', code='# Load the dataset\nurl = "https://raw.githubusercontent.com/pycaret/pycaret/master/datasets/diabetes.csv"\ndf = pd.read_csv(url)\n\n# Display basic information\nprint("Dataset shape:", df.shape)\nprint("Column names:", df.columns.tolist())\n\n# 1. Handle missing values (zeros in medical measurements)\n# Medical measurements that cannot be zero (except for \'Number of times pregnant\')\nzero_not_allowed_columns = [\n    \'Plasma glucose concentration a 2 hours in an oral glucose tolerance test\',\n    \'Diastolic blood pressure (mm Hg)\',\n    \'Triceps skin fold thickness (mm)\',\n    \'2-Hour serum insulin (mu U/ml)\',\n    \'Body mass i

In [23]:
code_solution.prefix

'This code preprocesses the diabetes dataset for machine learning by handling missing values, normalizing numerical features, and preparing the final dataset for model training.'

In [24]:
print(code_solution.imports)

import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split


In [25]:
print(code_solution.code)

# Load the dataset
url = "https://raw.githubusercontent.com/pycaret/pycaret/master/datasets/diabetes.csv"
df = pd.read_csv(url)

# Display basic information
print("Dataset shape:", df.shape)
print("Column names:", df.columns.tolist())

# 1. Handle missing values (zeros in medical measurements)
# Medical measurements that cannot be zero (except for 'Number of times pregnant')
zero_not_allowed_columns = [
    'Plasma glucose concentration a 2 hours in an oral glucose tolerance test',
    'Diastolic blood pressure (mm Hg)',
    'Triceps skin fold thickness (mm)',
    '2-Hour serum insulin (mu U/ml)',
    'Body mass index (weight in kg/(height in m)^2)'
]

# Replace 0s with NaN in columns where 0 is not a valid measurement
for column in zero_not_allowed_columns:
    df[column] = df[column].replace(0, np.nan)

# Check for missing values
print("\nMissing values after replacement:")
print(df.isnull().sum())

# Replace NaN values with the median of each column
for column in zero_not_allowed_co

In [10]:
from langgraph.graph import StateGraph, MessagesState

class State(MessagesState): # messages
    '''
    Represents the state of our graph.

    Attributes:
        error: Binary flag for control flow to indicate whether test error was tripped
        context: Data summary
        generation: Code solution
        iterations: Number of tries
    '''

    error: str # yer or no
    context: str
    generation: str
    iterations: int

graph_builder = StateGraph(State)

In [11]:
llm_with_tools = llm_gpt.bind_tools(tools=[describe_data])

In [12]:
def chatbot(state: State):
    print('##### HI ! #####')
    response = llm_with_tools.invoke(state['messages'])
    print('첫번째 LLM 호출 결과 : ', response)
    return {'messages': [response]}

graph_builder.add_node('chatbot', chatbot)

In [13]:
def add_context(state: State):
    print('##### ADD CONTEXT #####')
    if messages := state.get('messages', []):
        message = messages[-1]
    else:
        raise ValueError('No message found in input')

    for tool_call in message.tool_calls:
        for tool in tools:
            if tool.name == tool_call['name']:
                describe_str = tool.invoke(tool_call['args'])
    
    print('데이터 통계 (context) : ', describe_str[:100])
    return {'context': describe_str}

graph_builder.add_node('add_context', add_context)